# 0. Imports

In [42]:
import os
import random

import numpy as np

from constants import (
    G, GOAL_Y, GOAL_X_MIN, GOAL_X_MAX, FPS,
    THROW_ERROR_RADIUS, THROW_INIT_HEIGHT_MIN, THROW_INIT_HEIGHT_MAX,
    THROW_ANGLE_DEG_MIN, THROW_ANGLE_DEG_MAX,
)
from physics import (
    generate_ball_trajectory,
    compute_ball_position_at_time,
    dog_velocity,
)
from interception import compute_interception
from objects import DRAG_COEFFS, print_drag_table
from dog_profile import DogProfile
from visualization import plot


# 1. Throwable objects & drag coefficients

Each object has a mass, frontal area and shape drag coefficient. The drag term
used by the simulation is computed from those physical properties (it is **not**
hard-coded), in `objects.py`:

$$\text{drag} = \frac{c_d \cdot \rho \cdot A}{2 \cdot m}$$

In [43]:
print_drag_table()

Real-life simulated drag coefficients:
--------------------------------------------------
Ball            : 0.0181  (Mass: 0.058kg, Area: 0.0034m²)
Sausage         : 0.0226  (Mass:   0.1kg, Area: 0.0045m²)
Pizza           : 0.0735  (Mass:  0.14kg, Area: 0.0150m²)
Taco            : 0.0368  (Mass:  0.15kg, Area: 0.0090m²)
Steak           : 0.0408  (Mass:   0.3kg, Area: 0.0200m²)
Fry             : 0.1715  (Mass: 0.003kg, Area: 0.0004m²)


# 2. Dog profiles

Everything that depends on *which* dog is fetching lives in a `DogProfile`
dataclass (`dog_profile.py`). Its **defaults already describe the idealized
`OPTIMAL` dog**, so that profile needs no arguments. `FITZ` (error-prone dog from the problem) is built by overriding only the fields that differ: he is
slower, reacts later, and mistimes his jump and jaw.

In [44]:
# Idealized dog: fast, no mistiming
OPTIMAL = DogProfile(name="Optimal")

# Fitz, my real dog: slower and prone to mistiming
FITZ = DogProfile(
    name="Fitz",
    reaction_time_ms_min=250,
    reaction_time_ms_max=350,
    max_speed=8.0,
    acceleration=7.0,
    jaw_open_speed_multiplier=7.0,
    jump_mistime_min=-0.25,
    jump_mistime_max=0.25,
    jaw_error_min=-0.4,
    jaw_error_max=0.4,
    jaw_close_mistime_min=-0.15,
    jaw_close_mistime_max=0.0,
)

# The dog used by simulate() / run_batch() unless one is passed explicitly.
DOG = FITZ

OPTIMAL, FITZ

(DogProfile(name='Optimal', reaction_time_ms_min=150, reaction_time_ms_max=300, max_speed=12.0, acceleration=15.0, jaw_rotation_speed=7.0, arrow_length=0.12, arrow_height=0.07, jaw_open_angle=0.4, jaw_angle_offset=0.2, jaw_open_speed_multiplier=10.0, jaw_close_duration=0.05, jump_mistime_min=0.0, jump_mistime_max=0.0, jaw_error_min=0.0, jaw_error_max=0.0, jaw_close_mistime_min=0.0, jaw_close_mistime_max=0.0),
 DogProfile(name='Fitz', reaction_time_ms_min=250, reaction_time_ms_max=350, max_speed=8.0, acceleration=7.0, jaw_rotation_speed=7.0, arrow_length=0.12, arrow_height=0.07, jaw_open_angle=0.4, jaw_angle_offset=0.2, jaw_open_speed_multiplier=7.0, jaw_close_duration=0.05, jump_mistime_min=-0.25, jump_mistime_max=0.25, jaw_error_min=-0.4, jaw_error_max=0.4, jaw_close_mistime_min=-0.15, jaw_close_mistime_max=0.0))

# 3. Dog motion

Step the dog through the timeline: it reacts after its delay, runs toward the
intercept point (chosen by `compute_interception`), then jumps and eventually
snaps its jaw shut.

In [45]:
def _jump_kinematics(target_y):
    """Return (jump_velocity, time_to_apex) needed to reach ``target_y``."""
    jump_height = max(0, target_y - GOAL_Y)
    if jump_height > 0.01:
        jump_vy = np.sqrt(2 * G * jump_height)
        return jump_vy, jump_vy / G
    return 0.0, 0.0


def _advance_dog(current_pos, target, intercept_t, t_now, dt_frame,
                 dog_vx, dog_vy, dog_run_time, jump_mistime, dog):
    """Advance the dog one frame. Returns updated state tuple."""
    target_x, target_y = target
    dx = target_x - current_pos[0]
    dist_x = abs(dx)
    dog_jumped = False

    if current_pos[1] > 0.01:
        # Airborne: ballistic arc.
        new_x = current_pos[0] + dog_vx * dt_frame
        new_y = current_pos[1] + dog_vy * dt_frame - 0.5 * G * dt_frame**2
        dog_vy -= G * dt_frame
        if (dx < 0 and new_x < target_x) or (dx > 0 and new_x > target_x):
            new_x = target_x
            dog_vx = 0.0
        if new_y <= 0:
            new_y = 0.0
            dog_vy = 0.0
            dog_vx = 0.0
            dog_jumped = True
        return [new_x, new_y], dog_vx, dog_vy, dog_run_time, dog_jumped

    if dist_x > 0.02:
        # Running horizontally toward the target.
        dog_run_time += dt_frame
        speed = dog_velocity(dog_run_time, dog.acceleration, dog.max_speed)
        dog_vx = speed if dx > 0 else -speed

        jump_vy, time_to_apex = _jump_kinematics(target_y)
        time_remaining = intercept_t - t_now if intercept_t is not None else 999

        if jump_vy > 0 and time_remaining <= time_to_apex + jump_mistime:
            dog_vy = jump_vy
            new_x = current_pos[0] + dog_vx * dt_frame
            new_y = dog_vy * dt_frame
            return [new_x, max(0, new_y)], dog_vx, dog_vy, dog_run_time, dog_jumped

        new_x = current_pos[0] + dog_vx * dt_frame
        if (dx < 0 and new_x < target_x) or (dx > 0 and new_x > target_x):
            new_x = target_x
            dog_vx = 0.0
        return [new_x, 0.0], dog_vx, dog_vy, dog_run_time, dog_jumped

    # In position horizontally: wait, then jump.
    jump_vy, time_to_apex = _jump_kinematics(target_y)
    if jump_vy > 0:
        time_remaining = intercept_t - t_now if intercept_t is not None else 999
        if time_remaining <= time_to_apex + jump_mistime:
            dog_vy = jump_vy
            new_y = dog_vy * dt_frame
            return [current_pos[0], max(0, new_y)], dog_vx, dog_vy, dog_run_time, dog_jumped
        return list(current_pos), dog_vx, dog_vy, dog_run_time, dog_jumped

    return list(current_pos), dog_vx, dog_vy, dog_run_time, True


def simulate_dog_motion(dog, t, ball_t_vals, ball_x_vals, ball_y_vals,
                        t_flight, reaction_delay_s, goal_x, jump_mistime):
    """Simulate the dog over the timeline ``t``.

    Returns ``(dog_positions, intercept_target, intercept_t, intercept_is_q2)``.
    ``dog_positions`` is an ``(N, 2)`` array of (x, height-above-rest) points.
    """
    num_frames = len(t)
    dog_positions = np.zeros((num_frames, 2))
    dog_positions[0] = [goal_x, 0.0]

    dog_vx = dog_vy = 0.0
    intercept_target = intercept_t = None
    intercept_is_q2 = dog_started = dog_jumped = False
    dog_run_time = 0.0

    for i in range(1, num_frames):
        dt_frame = t[i] - t[i - 1]
        current_pos = dog_positions[i - 1]

        if t[i] >= reaction_delay_s and not dog_started:
            intercept_target, intercept_t, intercept_is_q2 = compute_interception(
                dog, current_pos[0], ball_t_vals, ball_x_vals, ball_y_vals, t_flight, t[i]
            )
            dog_started = True

        if dog_started and intercept_target is not None and not dog_jumped:
            dog_positions[i], dog_vx, dog_vy, dog_run_time, dog_jumped = _advance_dog(
                current_pos, intercept_target, intercept_t, t[i], dt_frame,
                dog_vx, dog_vy, dog_run_time, jump_mistime, dog,
            )
        else:
            dog_positions[i] = current_pos

    return dog_positions, intercept_target, intercept_t, intercept_is_q2


def compute_jaw_target(intercept_target, intercept_t,
                       ball_t_vals, ball_x_vals, ball_y_vals, jaw_error, dog):
    """Unit direction the jaw should aim at the intercept point."""
    if intercept_target is None or intercept_t is None:
        return (-1.0, 0.0)

    dt_v = 0.01
    bx1, by1 = compute_ball_position_at_time(intercept_t, ball_t_vals, ball_x_vals, ball_y_vals)
    bx2, by2 = compute_ball_position_at_time(intercept_t + dt_v, ball_t_vals, ball_x_vals, ball_y_vals)
    ball_vx = (bx2 - bx1) / dt_v
    ball_vy = (by2 - by1) / dt_v
    jaw_angle = np.arctan2(-ball_vy, -ball_vx) + dog.jaw_angle_offset + jaw_error
    return (np.cos(jaw_angle), np.sin(jaw_angle))

# 4. Jaw helpers & catch detection

Shared helpers describing where the jaw points and how open it is, plus the
check that decides whether the ball ends up inside the closing jaw.

In [46]:
_JAW_INIT_ANGLE = np.arctan2(0.0, -1.0)  # jaw initially points left


def rotated_jaw_angle(dog, jaw_target, time_since_seen):
    """Angle the jaw is pointing at, given how long it has been rotating."""
    target_angle = np.arctan2(jaw_target[1], jaw_target[0])
    angle_diff = (target_angle - _JAW_INIT_ANGLE + np.pi) % (2 * np.pi) - np.pi
    max_rotation = dog.jaw_rotation_speed * time_since_seen
    if abs(angle_diff) > 0.001:
        rotation = np.sign(angle_diff) * min(abs(angle_diff), max_rotation)
    else:
        rotation = 0
    return _JAW_INIT_ANGLE + rotation


def jaw_open_closing(dog, t_now, trigger_time):
    """Jaw opening half-angle during the closing animation (used for catching)."""
    if t_now < trigger_time:
        return dog.jaw_open_angle
    if t_now < trigger_time + dog.jaw_close_duration:
        close_progress = (t_now - trigger_time) / dog.jaw_close_duration
        return dog.jaw_open_angle * (1.0 - close_progress)
    return 0.0


def check_catch_success(dog, t, dog_positions, intercept_target, intercept_t,
                        jaw_target, reaction_delay_s,
                        ball_t_vals, ball_x_vals, ball_y_vals, jaw_close_mistime):
    """Return ``True`` if the ball falls within the jaw cone while it is open."""
    if intercept_target is None or intercept_t is None:
        return False

    trigger_time = intercept_t + jaw_close_mistime
    check_times = np.linspace(intercept_t - 0.05, intercept_t + 0.05, 20)
    for t_check in check_times:
        if t_check < reaction_delay_s:
            continue

        dog_x = np.interp(t_check, t, dog_positions[:, 0])
        dog_y = np.interp(t_check, t, dog_positions[:, 1])
        head_x = dog_x
        head_y = dog_y + GOAL_Y

        t_ball = min(t_check, ball_t_vals[-1])
        ball_x, ball_y = compute_ball_position_at_time(t_ball, ball_t_vals, ball_x_vals, ball_y_vals)

        dx = ball_x - head_x
        dy = ball_y - head_y
        if np.sqrt(dx**2 + dy**2) > dog.arrow_length:
            continue

        current_angle = rotated_jaw_angle(dog, jaw_target, t_check - reaction_delay_s)
        angle_to_ball = np.arctan2(dy, dx)
        angle_from_jaw = (angle_to_ball - current_angle + np.pi) % (2 * np.pi) - np.pi

        if abs(angle_from_jaw) <= jaw_open_closing(dog, t_check, trigger_time) + 0.01:
            return True

    return False

# 5. High-level simulation

Tie everything together: generate a throw, run the chosen dog, and either
animate the attempt (`simulate`) or just report success (`simulate_silent`).

In [47]:
def _setup_throw(dog):
    """Randomly generate a throw for ``dog``. Returns a dict of derived quantities or None."""
    goal_x = random.uniform(GOAL_X_MIN, GOAL_X_MAX)
    thrown_object = random.choice(list(DRAG_COEFFS.keys()))
    drag = DRAG_COEFFS[thrown_object]

    jump_mistime = dog.sample_jump_mistime()
    jaw_error = dog.sample_jaw_error()
    jaw_close_mistime = dog.sample_jaw_close_mistime()

    r_offset = THROW_ERROR_RADIUS * np.sqrt(random.uniform(0, 1))
    theta_offset = random.uniform(0, 2 * np.pi)
    throw_target_x = goal_x + r_offset * np.cos(theta_offset)
    throw_target_y = GOAL_Y + r_offset * np.sin(theta_offset)

    throw_height = random.uniform(THROW_INIT_HEIGHT_MIN, THROW_INIT_HEIGHT_MAX)
    angle_deg = random.uniform(THROW_ANGLE_DEG_MIN, THROW_ANGLE_DEG_MAX)
    angle = np.radians(angle_deg)

    numerator = G * (throw_target_x**2)
    denominator = 2 * (np.cos(angle) ** 2) * (
        throw_height - throw_target_y + throw_target_x * np.tan(angle)
    )
    if denominator <= 0:
        return None
    v0 = np.sqrt(numerator / denominator)

    vx = v0 * np.cos(angle)
    vy = v0 * np.sin(angle)
    ball_t_vals, ball_x_vals, ball_y_vals = generate_ball_trajectory(vx, vy, throw_height, G, drag)

    return {
        "goal_x": goal_x,
        "thrown_object": thrown_object,
        "jump_mistime": jump_mistime,
        "jaw_error": jaw_error,
        "jaw_close_mistime": jaw_close_mistime,
        "angle_deg": angle_deg,
        "v0": v0,
        "ball_t_vals": ball_t_vals,
        "ball_x_vals": ball_x_vals,
        "ball_y_vals": ball_y_vals,
        "t_flight": ball_t_vals[-1],
        "throw_range": ball_x_vals[-1],
        "reaction_delay_s": dog.sample_reaction_delay_s(),
    }


def _build_timeline(setup):
    """Build the frame timeline for an attempt."""
    t_total = setup["t_flight"] + setup["reaction_delay_s"] + 0.3
    num_frames = max(2, int(t_total * FPS))
    return np.linspace(0, t_total, num_frames), num_frames


def _run_attempt(dog, setup):
    """Run the dog motion and jaw computation for a prepared throw."""
    t, num_frames = _build_timeline(setup)

    dog_positions, intercept_target, intercept_t, intercept_is_q2 = simulate_dog_motion(
        dog, t, setup["ball_t_vals"], setup["ball_x_vals"], setup["ball_y_vals"],
        setup["t_flight"], setup["reaction_delay_s"], setup["goal_x"], setup["jump_mistime"],
    )
    jaw_target = compute_jaw_target(
        intercept_target, intercept_t,
        setup["ball_t_vals"], setup["ball_x_vals"], setup["ball_y_vals"],
        setup["jaw_error"], dog,
    )
    return t, num_frames, dog_positions, intercept_target, intercept_t, intercept_is_q2, jaw_target


def simulate(seed=None, dog=DOG, video_dir='videos'):
    """Run a single attempt and save its animation as an MP4"""
    if seed is None:
        seed = random.randint(0, 2**31)
    random.seed(seed)
    np.random.seed(seed)

    setup = _setup_throw(dog)
    if setup is None:
        return None

    (t, num_frames, dog_positions, intercept_target, intercept_t,
     intercept_is_q2, jaw_target) = _run_attempt(dog, setup)

    t1 = np.clip(t, 0, setup["t_flight"])
    x1 = np.interp(t1, setup["ball_t_vals"], setup["ball_x_vals"])
    y1 = np.interp(t1, setup["ball_t_vals"], setup["ball_y_vals"])

    t2 = np.clip(t - setup["reaction_delay_s"], 0, setup["t_flight"])
    x2 = np.interp(t2, setup["ball_t_vals"], setup["ball_x_vals"])
    y2 = np.interp(t2, setup["ball_t_vals"], setup["ball_y_vals"])

    save_path = os.path.join(
        video_dir, f"{dog.name.lower()}_{setup['thrown_object']}_{seed}.mp4"
    )
    plot(
        dog, rotated_jaw_angle, setup["throw_range"], setup["angle_deg"], setup["goal_x"],
        THROW_ERROR_RADIUS, setup["v0"], x1, y1, x2, y2, t,
        setup["reaction_delay_s"], num_frames, dog_positions,
        intercept_target, intercept_is_q2, intercept_t, jaw_target,
        setup["thrown_object"], setup["jaw_close_mistime"], save_path
    )
    print(f"Saved {save_path}")
    return save_path


def simulate_silent(seed=None, dog=DOG):
    """Run a single attempt without plotting. Returns (success, seed, object)."""
    if seed is None:
        seed = random.randint(0, 2**31)
    random.seed(seed)
    np.random.seed(seed)

    setup = _setup_throw(dog)
    if setup is None:
        return None

    (t, _num_frames, dog_positions, intercept_target, intercept_t,
     _intercept_is_q2, jaw_target) = _run_attempt(dog, setup)

    success = check_catch_success(
        dog, t, dog_positions, intercept_target, intercept_t, jaw_target,
        setup["reaction_delay_s"], setup["ball_t_vals"], setup["ball_x_vals"],
        setup["ball_y_vals"], setup["jaw_close_mistime"],
    )
    return success, seed, setup["thrown_object"]


def run_batch(n=1000, show_failures=5, dog=DOG, batch_seed=0):
    """Run ``n`` silent attempts for ``dog``, print stats, and replay some failures.

    The ``batch_seed`` deterministically generates the list of per-attempt
    seeds, so two calls of ``run_batch`` with the same ``batch_seed`` run the
    exact same set of throws -- which lets you compare e.g. Optimal vs Fitz on
    identical inputs. Pass ``batch_seed=None`` for a non-reproducible run.
    """
    if batch_seed is None:
        attempt_seeds = [random.randint(0, 2**31) for _ in range(n)]
    else:
        rng = random.Random(batch_seed)
        attempt_seeds = [rng.randint(0, 2**31) for _ in range(n)]

    successes = valid = 0
    failure_seeds = []
    stats = {k: {"success": 0, "total": 0} for k in DRAG_COEFFS}

    for attempt_seed in attempt_seeds:
        result = simulate_silent(seed=attempt_seed, dog=dog)
        if result is None:
            continue
        success, seed, thrown_object = result
        valid += 1
        stats[thrown_object]["total"] += 1
        if success:
            successes += 1
            stats[thrown_object]["success"] += 1
        elif len(failure_seeds) < show_failures:
            failure_seeds.append(seed)

    print(f"[{dog.name}] Overall Success: {successes}/{valid} "
          f"({100 * successes / max(valid, 1):.1f}%)")
    for obj, data in stats.items():
        if data["total"] > 0:
            rate = 100 * data["success"] / data["total"]
            failed = data["total"] - data["success"]
            print(f"  {obj.capitalize()} Success Rate: {rate:.1f}% "
                  f"({failed}/{data['total']} failed)")

    print(f"\nShowing {len(failure_seeds)} failures:\n")
    for seed in failure_seeds:
        simulate(seed=seed, dog=dog)

# 6. Run it

Batch statistics for Fitz, then a single animated attempt. Each call to
`simulate` renders the animation and **saves it as an MP4 in the `videos/`
directory**.


In [48]:
run_batch(1000, dog=FITZ)

[Fitz] Overall Success: 208/1000 (20.8%)
  Ball Success Rate: 25.1% (143/191 failed)
  Sausage Success Rate: 21.2% (130/165 failed)
  Pizza Success Rate: 18.8% (138/170 failed)
  Taco Success Rate: 24.7% (122/162 failed)
  Steak Success Rate: 19.5% (128/159 failed)
  Fry Success Rate: 14.4% (131/153 failed)

Showing 5 failures:

Saved videos/fitz_steak_1654615998.mp4
Saved videos/fitz_steak_1806341205.mp4
Saved videos/fitz_fry_173879092.mp4
Saved videos/fitz_ball_1112038970.mp4
Saved videos/fitz_pizza_2087043557.mp4


In [49]:
simulate(dog=FITZ)

Saved videos/fitz_steak_954827791.mp4


'videos/fitz_steak_954827791.mp4'

For comparison, the idealized dog:

In [50]:
run_batch(1000, dog=OPTIMAL)

[Optimal] Overall Success: 919/1000 (91.9%)
  Ball Success Rate: 92.7% (14/191 failed)
  Sausage Success Rate: 92.7% (12/165 failed)
  Pizza Success Rate: 88.8% (19/170 failed)
  Taco Success Rate: 93.2% (11/162 failed)
  Steak Success Rate: 88.7% (18/159 failed)
  Fry Success Rate: 95.4% (7/153 failed)

Showing 5 failures:

Saved videos/optimal_steak_1806341205.mp4
Saved videos/optimal_pizza_1739178872.mp4
Saved videos/optimal_pizza_1712934065.mp4
Saved videos/optimal_pizza_391769539.mp4
Saved videos/optimal_steak_536057929.mp4
